# WAFER-GUARD · Fab-7 Excursion Detector — Audited & Repaired Pipeline

**Role:** Final Technical Reviewer on the Fab-7 Release Board  
**Auditor:** Anmol Suwalka  
**Status:** Repaired & Audited Pipeline (Pre-Release Audit)  
**Deliverable:** Corrected PyTorch Notebook (`wafer_guard_fixed.ipynb`)  



## Executive Summary & Audit Overview

The initial candidate model reported an inflated $F1 \ge 0.90$ ($0.9915$). Our audit proved this metric was entirely spurious, driven by three silent fatal defects:
1. **Target & Causality Leakage**: Global imputation, global scaling, and SMOTE applied to the entire dataset prior to splitting, evaluating on an artificial 50% defect prevalence and leaking future data into past predictions.
2. **Loss Gradient Corruption via PyTorch Broadcasting**: Tensor shape mismatch (`reconstructed` is `[64, 590]` vs `batch_x` is `[64, 1, 590]`) silently evaluated $(B \times B)$ cross-sample differences (`[64, 64, 590]`), pulling wafer representations toward the batch centroid.
3. **Domain Invariant Blindness (Legacy Probe Station PS-L)**: Sentinel values (`sensor_0 == 0.0`) and categorical firmware error codes (`sensor_1 \in \{100, 200, 300\}`) were treated as continuous physical measurements and scaled with continuous data.

Below is the fully repaired, causality-safe end-to-end pipeline evaluated at the true natural wafer defect prevalence (~5.4% - 6.6%).



## 0 · Environment Setup & Seeding


In [1]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             precision_recall_curve)
import joblib

warnings.filterwarnings("ignore")

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HP = dict(
    batch_size=64, epochs=40, lr=1e-3, weight_decay=1e-4,
    hidden=256, latent=32, lambda_clf=1.0, test_size=0.20
)
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")



PyTorch 2.13.0+cpu | Device: cpu


## 1 · Data Ingestion & Causal Ordering

Semiconductor fabrication is a continuous physical manufacturing process. Equipment ages, chambers undergo preventive maintenance, and recipes drift over time.
To enforce **strict causality**, we parse the `timestamp` column and sort the dataset chronologically. No future time $t' > t$ may influence any training step or historical prediction.



In [2]:
df = pd.read_csv("secom_telemetry_modified.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

y_raw = (df["label"].to_numpy() == 1).astype(np.int64)  # 1 = defective wafer

print(f"Dataset shape: {df.shape}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Natural defect prevalence: {y_raw.mean():.2%} ({y_raw.sum()} fail / {(1 - y_raw).sum()} pass)")



Dataset shape: (1567, 592)
Time range: 2008-07-19 11:55:00 to 2008-10-17 06:07:00
Natural defect prevalence: 6.64% (104 fail / 1463 pass)


## 2 · Domain Logic: Handling Legacy Probe Station PS-L Invariants

**Domain Context:**
A fraction of wafers are routed through legacy *Probe Station PS-L*. For these wafers:
- `sensor_0` contains a sentinel value `0.0` (not a physical sensor measurement).
- `sensor_1` contains discrete categorical firmware error codes: `100.0`, `200.0`, or `300.0`.

**Repair Strategy:**
1. Isolate the legacy routing flag: `is_legacy = (df['sensor_0'] == 0.0)`.
2. Extract the categorical error codes into one-hot binary indicator features (`ps_l_err_100`, `ps_l_err_200`, `ps_l_err_300`).
3. For continuous feature processing, mask out legacy `sensor_0` and `sensor_1` with `NaN` so continuous scaling statistics are not contaminated by non-continuous signals.



In [3]:
# 1. Detect legacy Probe Station PS-L wafers
is_legacy = (df["sensor_0"] == 0.0).astype(np.float32)
print(f"Legacy Probe Station PS-L wafers detected: {int(is_legacy.sum())} / {len(df)} ({is_legacy.mean():.2%})")

# 2. Extract categorical error codes
err_100 = ((is_legacy == 1.0) & (df["sensor_1"] == 100.0)).astype(np.float32)
err_200 = ((is_legacy == 1.0) & (df["sensor_1"] == 200.0)).astype(np.float32)
err_300 = ((is_legacy == 1.0) & (df["sensor_1"] == 300.0)).astype(np.float32)

cat_df = pd.DataFrame({
    "ps_l_legacy_station": is_legacy,
    "ps_l_err_100": err_100,
    "ps_l_err_200": err_200,
    "ps_l_err_300": err_300
})
print("Categorical error code distribution:")
print(f"  Error 100.0: {int(err_100.sum())} wafers")
print(f"  Error 200.0: {int(err_200.sum())} wafers")
print(f"  Error 300.0: {int(err_300.sum())} wafers")

# 3. Continuous feature isolation
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
df_cont = df[sensor_cols].copy()
df_cont.loc[is_legacy == 1.0, "sensor_0"] = np.nan
df_cont.loc[is_legacy == 1.0, "sensor_1"] = np.nan



Legacy Probe Station PS-L wafers detected: 146 / 1567 (9.32%)
Categorical error code distribution:
  Error 100.0: 61 wafers
  Error 200.0: 35 wafers
  Error 300.0: 50 wafers


## 3 · Causality-Strict Chronological Partitioning & Preprocessing

To guarantee zero data leakage:
1. **Chronological Split**: Partition the first 80% of lots as training ($t \le T_{split}$) and the remaining 20% as out-of-sample production test ($t > T_{split}$).
2. **In-Sample Fitting**: Imputer and Scaler are fitted **strictly on the training partition**. Zero evaluation statistics leak into preprocessing.
3. **No Test Resampling**: The test set remains untouched at its natural defect prevalence (~5.4% - 6.6%).



In [4]:
N = len(df)
split_idx = int(N * (1 - HP["test_size"]))

train_cont = df_cont.iloc[:split_idx].copy()
test_cont  = df_cont.iloc[split_idx:].copy()

train_cat = cat_df.iloc[:split_idx].to_numpy()
test_cat  = cat_df.iloc[split_idx:].to_numpy()

y_train = y_raw[:split_idx]
y_test  = y_raw[split_idx:]

print(f"Train lots: {len(train_cont)} (Defects: {y_train.sum()} / {len(y_train)} = {y_train.mean():.2%})")
print(f"Test lots:  {len(test_cont)} (Defects: {y_test.sum()} / {len(y_test)} = {y_test.mean():.2%})")

# Drop continuous sensors that are constant (zero-variance) in training
col_stds = train_cont.std(skipna=True)
valid_cols = col_stds[col_stds > 1e-5].index.tolist()
print(f"Active non-constant continuous features: {len(valid_cols)} / {len(sensor_cols)}")

# Fit imputer and scaler ONLY on training data
imputer = SimpleImputer(strategy="median")
X_tr_imp = imputer.fit_transform(train_cont[valid_cols])
X_te_imp = imputer.transform(test_cont[valid_cols])

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_imp)
X_te_scaled = scaler.transform(X_te_imp)

# Combine continuous channels with legacy categorical indicators
X_train_full = np.hstack([X_tr_scaled, train_cat])
X_test_full  = np.hstack([X_te_scaled, test_cat])

F_dim = X_train_full.shape[1]
print(f"Final feature dimension: {F_dim} channels")



Train lots: 1253 (Defects: 87 / 1253 = 6.94%)
Test lots:  314 (Defects: 17 / 314 = 5.41%)
Active non-constant continuous features: 468 / 590
Final feature dimension: 472 channels


## 4 · Tensors, Loaders & Exact Dimension Alignment

In the flawed notebook, an explicit channel dimension `[B, 1, F]` combined with a decoder output of `[B, F]` caused silent broadcasting into a `[B, B, F]` tensor.
Here, all tensors are strictly 2D `[B, F]`, ensuring exact single-wafer reconstruction and mathematically pure gradient computation.



In [5]:
def make_clean_loader(X, y, shuffle, batch_size=HP["batch_size"]):
    # Tensors shaped strictly [N, F] — no extra dimensions
    Xt = torch.from_numpy(X).float()
    yt = torch.from_numpy(y).float()
    g = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size,
                      shuffle=shuffle, generator=g, drop_last=False)

train_loader = make_clean_loader(X_train_full, y_train, shuffle=True)
test_loader  = make_clean_loader(X_test_full,  y_test,  shuffle=False)
print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")



Train batches: 20 | Test batches: 5


## 5 · Supervised Autoencoder Architecture

The repaired `RobustWaferSAE` combines:
1. An **Autoencoder Backbone** that learns a compact 32-dimensional latent representation $z$.
2. An **Aligned Decoder** reconstructing $x$ to minimize per-wafer reconstruction loss.
3. A **Calibrated Classification Head** mapping $z \to \text{logit}$, trained with class-weighted loss to handle extreme class imbalance causally.



In [6]:
class RobustWaferSAE(nn.Module):
    def __init__(self, d_in: int, d_hid: int = 256, d_lat: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hid), nn.LayerNorm(d_hid), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(d_hid, d_hid // 2), nn.LayerNorm(d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_lat),
        )
        self.decoder = nn.Sequential(
            nn.Linear(d_lat, d_hid // 2), nn.GELU(),
            nn.Linear(d_hid // 2, d_hid), nn.GELU(),
            nn.Linear(d_hid, d_in),
        )
        self.head = nn.Sequential(
            nn.Linear(d_lat, 16),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), self.head(z).squeeze(-1), z

model = RobustWaferSAE(F_dim, HP["hidden"], HP["latent"]).to(DEVICE)
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters()):,}")



Total trainable parameters: 317,977


## 6 · Training with Corrected Tensor Mechanics & Class Weighting

To handle the natural defect prevalence without synthesizing synthetic test data, we compute an exact positive class weight:
$$\text{pos\_weight} = \frac{N_{\text{normal}}}{N_{\text{defective}}}$$
Reconstruction loss is evaluated using strictly aligned shapes $[B, F]$, completely eliminating cross-sample broadcast corruption.



In [7]:
pos_ratio = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
pos_weight = torch.tensor([pos_ratio], dtype=torch.float32).to(DEVICE)
print(f"Causally computed pos_weight: {pos_weight.item():.2f}")

opt   = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=HP["epochs"])
bce   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

history = {"total": [], "recon": [], "clf": []}

for epoch in range(1, HP["epochs"] + 1):
    model.train()
    acc = {"total": 0.0, "recon": 0.0, "clf": 0.0}
    n_batches = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)

        reconstructed, logits, _ = model(batch_x)

        # EXACT SHAPE MATCH [B, F] — Zero broadcasting corruption!
        recon_loss = nn.functional.mse_loss(reconstructed, batch_x)
        clf_loss   = bce(logits, batch_y)
        loss       = recon_loss + HP["lambda_clf"] * clf_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        acc["total"] += loss.item()
        acc["recon"] += recon_loss.item()
        acc["clf"]   += clf_loss.item()
        n_batches += 1

    sched.step()
    for k in acc:
        history[k].append(acc[k] / n_batches)
    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:02d}/{HP['epochs']} | Total: {history['total'][-1]:.4f} "
              f"| Recon MSE: {history['recon'][-1]:.4f} | Clf BCE: {history['clf'][-1]:.4f}")



Causally computed pos_weight: 13.40
Epoch 01/40 | Total: 2.2717 | Recon MSE: 0.9902 | Clf BCE: 1.2815
Epoch 05/40 | Total: 1.1028 | Recon MSE: 0.8954 | Clf BCE: 0.2074
Epoch 10/40 | Total: 0.8036 | Recon MSE: 0.7829 | Clf BCE: 0.0207
Epoch 15/40 | Total: 0.7154 | Recon MSE: 0.7046 | Clf BCE: 0.0107
Epoch 20/40 | Total: 0.6494 | Recon MSE: 0.6480 | Clf BCE: 0.0014
Epoch 25/40 | Total: 0.6059 | Recon MSE: 0.6051 | Clf BCE: 0.0008
Epoch 30/40 | Total: 0.5790 | Recon MSE: 0.5782 | Clf BCE: 0.0008
Epoch 35/40 | Total: 0.5694 | Recon MSE: 0.5686 | Clf BCE: 0.0008
Epoch 40/40 | Total: 0.5649 | Recon MSE: 0.5642 | Clf BCE: 0.0007


## 7 · Decision Threshold Calibration & Natural Prevalence Evaluation

In highly imbalanced production environments ($94.6\%$ pass vs $5.4\%$ fail), a default $0.50$ probability threshold is sub-optimal.
We calibrate the decision threshold $\tau^*$ on training lot history to maximize the F1 / precision-recall trade-off, then evaluate on the unseen out-of-sample test partition at **natural prevalence**.



In [8]:
model.eval()

# 1. Calibrate threshold on training history
with torch.no_grad():
    tr_x = torch.from_numpy(X_train_full).float().to(DEVICE)
    _, tr_logits, _ = model(tr_x)
    tr_probs = torch.sigmoid(tr_logits).cpu().numpy()

best_th = 0.50
best_tr_f1 = 0.0
for th in np.linspace(0.10, 0.90, 81):
    preds = (tr_probs >= th).astype(int)
    f = f1_score(y_train, preds, zero_division=0)
    if f > best_tr_f1:
        best_tr_f1 = f
        best_th = th

print(f"Causally tuned decision threshold: {best_th:.2f} (Training F1: {best_tr_f1:.4f})")

# 2. Out-of-sample Test Evaluation at Natural Prevalence (~5.41%)
test_probs = []
with torch.no_grad():
    for xb, _ in test_loader:
        _, logits, _ = model(xb.to(DEVICE))
        test_probs.append(torch.sigmoid(logits).cpu())
test_probs = torch.cat(test_probs).numpy()

final_preds = (test_probs >= best_th).astype(int)

test_f1   = f1_score(y_test, final_preds, zero_division=0)
test_prec = precision_score(y_test, final_preds, zero_division=0)
test_rec  = recall_score(y_test, final_preds, zero_division=0)
test_auc  = roc_auc_score(y_test, test_probs)
test_ap   = average_precision_score(y_test, test_probs)
tn, fp, fn, tp = confusion_matrix(y_test, final_preds).ravel()

print("\n" + "="*60)
print(f"HONEST METRICS AT NATURAL PREVALENCE ({y_test.mean():.2%})")
print("="*60)
print(f"Decision Threshold:            {best_th:.2f}")
print(f"F1 Score:                      {test_f1:.4f}")
print(f"Precision:                     {test_prec:.4f}")
print(f"Recall:                        {test_rec:.4f}")
print(f"ROC-AUC:                       {test_auc:.4f}")
print(f"Average Precision (PR-AUC):    {test_ap:.4f} (Baseline: {y_test.mean():.4f})")
print(f"Confusion Matrix:              TN={tn} | FP={fp} | FN={fn} | TP={tp}")
print("="*60)



Causally tuned decision threshold: 0.10 (Training F1: 1.0000)

HONEST METRICS AT NATURAL PREVALENCE (5.41%)
Decision Threshold:            0.10
F1 Score:                      0.2143
Precision:                     0.2727
Recall:                        0.1765
ROC-AUC:                       0.6950
Average Precision (PR-AUC):    0.1257 (Baseline: 0.0541)
Confusion Matrix:              TN=289 | FP=8 | FN=14 | TP=3


## 8 · Release Gate Verdict & Production Recommendations

The release gate requires $F1 \ge 0.90$. We evaluate whether the repaired model meets the production standard.



In [9]:
GATE_F1 = 0.90
verdict = "PASSED ✅" if test_f1 >= GATE_F1 else "NO-GO 🛑 (REVIEW REQUIRED)"
print(f"Release Gate (F1 >= {GATE_F1}): {test_f1:.4f} -> {verdict}")

if test_f1 < GATE_F1:
    print("\n[CRITICAL AUDIT NOTICE]")
    print("The model fails the F1 >= 0.90 gate under honest, causality-safe evaluation.")
    print("Immediate unmonitored deployment to Fab-7 line monitoring must be HALTED.")
    print("Recommended path to production: Run model in Shadow Mode and evaluate ensemble approaches.")

# Save serialized artifacts
os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/wafer_sae_fixed.pt")
joblib.dump({"imputer": imputer, "scaler": scaler, "valid_cols": valid_cols, "threshold": best_th}, "artifacts/preprocessor.joblib")
print("\nRepaired artifacts written to ./artifacts/ successfully.")



Release Gate (F1 >= 0.9): 0.2143 -> NO-GO 🛑 (REVIEW REQUIRED)

[CRITICAL AUDIT NOTICE]
The model fails the F1 >= 0.90 gate under honest, causality-safe evaluation.
Immediate unmonitored deployment to Fab-7 line monitoring must be HALTED.
Recommended path to production: Run model in Shadow Mode and evaluate ensemble approaches.

Repaired artifacts written to ./artifacts/ successfully.
